# Demo 6, T2: A custom MCP server on Databricks Apps

**Scope**: turn `search_wiki` into an MCP server, run it as a Databricks App, and call it by hand. Nothing else: the agent does not use the server yet (that is T3).

In T1 the tools were in the agent's code. A custom MCP server is a separate program that any MCP client can ask "what tools do you have?" (`tools/list`) and "run this one" (`tools/call`). On Databricks it runs as an **app** [1].

What Session 1's `servers.py` changes to run there:

| Requirement | Session 1 (localhost) | Databricks Apps |
|---|---|---|
| Transport | STDIO, or HTTP | **Streamable HTTP** [1] |
| Host and port | `127.0.0.1:8765` | **`0.0.0.0`**, port from **`DATABRICKS_APP_PORT`** (default `8000`) [1] |
| Path | any | Whatever the server serves, and clients are given that path. **`/mcp`** is the SDK's default and what Databricks' templates use [1] |
| Name | any | Starts with **`mcp-`**, so AI Playground recognizes it [1] |
| Who may call | anyone on the machine | **`CAN USE`** on the app, signed in with **OAuth**. Personal access tokens are refused [2] |

**Before running**: the `mcp-wiki-server/` folder must sit next to `common/`, you need permission to create Databricks Apps, and `mcp-wiki-server/requirements.txt` must pin `mcp<2` (T2.1 checks this). For T2.3 you need a service principal and an OAuth secret for it, which an admin creates. [n] refers to **References** at the end.

## T2.0 Setup

In [0]:
%pip install -U databricks-sdk requests
dbutils.library.restartPython()

In [0]:
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path

import requests
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound, ResourceAlreadyExists
from databricks.sdk.service.apps import App, AppAccessControlRequest, AppDeployment, AppPermissionLevel

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from common.config import TOP_K

w = WorkspaceClient()
COMMON_DIR = Path(os.getcwd()).parent / "common"
APP_DIR = Path(os.getcwd()).parent / "mcp-wiki-server"
APP_NAME = "mcp-wiki-server"
SECRET_SCOPE = "mcp-demo"
WORKSPACE_PATH = f"/Workspace/Users/{w.current_user.me().user_name}/{APP_NAME}"
assert APP_NAME.startswith("mcp-"), "App names for MCP servers start with `mcp-`"


def app_mcp_url():
    """The MCP URL of the app if it exists, else None. `/mcp` is the path our server serves."""
    try:
        return f"{w.apps.get(APP_NAME).url.rstrip('/')}/mcp"
    except NotFound:
        return None


MCP_URL = app_mcp_url()  # set again after deploying in T2.2; already set if the app exists from an earlier run

HEADERS = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream"}


def rpc(method, params=None, *, url=None, auth=None, id=1):
    """One MCP request, by hand: a JSON-RPC message in an HTTP POST. Redirects are not followed, so we see them."""
    body = {"jsonrpc": "2.0", "id": id, "method": method}
    if params is not None:
        body["params"] = params
    return requests.post(url or MCP_URL, json=body, headers={**HEADERS, **(auth or {})}, allow_redirects=False, timeout=60)


def show(r, limit=1500):
    print("HTTP", r.status_code, "|", r.headers.get("content-type", ""))
    if r.headers.get("Location"):
        print("redirects to:", r.headers["Location"][:100])
    try:
        print(json.dumps(r.json(), indent=2)[:limit])
    except ValueError:
        print(r.text[:300])


print("App:", APP_NAME, "| MCP URL:", MCP_URL or "(the app does not exist yet; T2.2 creates it)")

App: mcp-wiki-server | MCP URL: https://mcp-wiki-server-4475275120697622.aws.databricksapps.com/mcp


## T2.1 The server

The tool is the same function as in T1, now behind the MCP Python SDK.

In [0]:
print((APP_DIR / "server.py").read_text())

"""
The tutor's `search_wiki` as an MCP server, to run as a Databricks App (06_tools.py, T2).

Session 1's server ran on 127.0.0.1. On Databricks Apps the same server needs four things:
    1. Streamable HTTP as the transport
    2. Bind to 0.0.0.0, on the port in DATABRICKS_APP_PORT (default 8000)
    3. An MCP path: `/mcp`, which is the SDK's default
    4. An app name that starts with `mcp-` (set when the app is created)
"""

import os
from typing import Annotated

from mcp.server.fastmcp import FastMCP
from pydantic import Field

from wiki_search import search_wiki as _search_wiki

mcp = FastMCP(
    "python-wiki",
    host="0.0.0.0",
    port=int(os.environ.get("DATABRICKS_APP_PORT", "8000")),
    # No sessions: every request stands alone, so there is no per-client state to lose between calls.
    stateless_http=True,
    # Plain JSON replies instead of event streams: simpler to call and to test.
    json_response=True,
)


@mcp.tool()
def search_wiki(query: Annotated[str, Field(d

**Look at**: `host="0.0.0.0"`, the port from `DATABRICKS_APP_PORT` and `transport="streamable-http"` are the Databricks requirements [1]. `/mcp` is the SDK's default path, which we leave alone. The tool's description and parameter description are the ones the model read in T1.

`stateless_http=True` (no sessions) and `json_response=True` (plain JSON replies) are our choices, not Databricks'. They are why T2.4 can call the server with one plain POST: a stateful server would first need an `initialize` request and a session id.

The server file imports `mcp.server.fastmcp`. **MCP Python SDK 2.x renamed `FastMCP`** (to `MCPServer`) [4], so the app's `requirements.txt` must keep the SDK below 2. Otherwise a redeploy that resolves a new version fails when the server starts. Check that here:

In [0]:
reqs = (APP_DIR / "requirements.txt").read_text()
print(reqs)

mcp_lines = [l.strip() for l in reqs.splitlines() if l.strip().lower().startswith("mcp")]
pinned = any(("<2" in l.replace(" ", "")) or ("==1." in l.replace(" ", "")) or ("~=1." in l.replace(" ", "")) for l in mcp_lines)
if pinned:
    print("OK: the MCP SDK is pinned below 2.")
else:
    print("WARNING: requirements.txt does not pin the MCP SDK below 2 (for example `mcp<2`). server.py imports FastMCP, which 2.x does not have.")

mcp>=1.24,<2

OK: the MCP SDK is pinned below 2.


## T2.2 Deploy it

Stage the app folder (server, `requirements.txt`, the search code, the wiki pages, an `app.yaml` that sets the start command and passes `TOP_K`), copy it to a workspace folder, create the app, deploy from the folder [3].

In [0]:
# app.yaml sets WIKI_TOP_K; the server only honours it if the search code reads that name.
if "WIKI_TOP_K" not in (COMMON_DIR / "wiki_search.py").read_text():
    print("WARNING: common/wiki_search.py does not mention WIKI_TOP_K, so the setting in app.yaml has no effect.")

stage = Path(tempfile.mkdtemp())
try:
    for src in [APP_DIR / "server.py", APP_DIR / "requirements.txt", COMMON_DIR / "wiki_search.py", COMMON_DIR / "wiki_pages.py"]:
        shutil.copy(src, stage / src.name)
    (stage / "app.yaml").write_text(f"command: ['python', 'server.py']\nenv:\n  - name: WIKI_TOP_K\n    value: '{TOP_K}'\n")
    shutil.copytree(stage, WORKSPACE_PATH, dirs_exist_ok=True)
    print((stage / "app.yaml").read_text())
finally:
    shutil.rmtree(stage, ignore_errors=True)

command: ['python', 'server.py']
env:
  - name: WIKI_TOP_K
    value: '2'



In [0]:
try:
    w.apps.get(APP_NAME)
except NotFound:
    w.apps.create_and_wait(app=App(name=APP_NAME, description="Python tutor: search_wiki as an MCP server"))

deployment = w.apps.deploy_and_wait(app_name=APP_NAME, app_deployment=AppDeployment(source_code_path=WORKSPACE_PATH))
print("Deployment:", deployment.status.state, "|", deployment.status.message)

app = w.apps.get(APP_NAME)
print("App status:", app.app_status.state, "|", app.app_status.message)

MCP_URL = app_mcp_url()
print("MCP server URL:", MCP_URL)

Deployment: AppDeploymentState.SUCCEEDED | App started successfully
App status: ApplicationState.RUNNING | App has status: App is running
MCP server URL: https://mcp-wiki-server-4475275120697622.aws.databricksapps.com/mcp


**Look at**: **Compute > Apps > mcp-wiki-server**: status, URL, logs [1]. If the app is not `RUNNING`, the logs say why.

> **If the deploy fails** with a permission error on the folder: the app runs as its own service principal (`app.service_principal_name`). Give it `CAN READ` on the folder and deploy again [3].
>
> **If the app starts and then stops** with an import error for `mcp.server.fastmcp`: the SDK resolved to 2.x. Pin `mcp<2` in `requirements.txt` and deploy again (T2.1).

The app keeps running, and billing under Databricks Apps pricing [1], until it is stopped. T3 to T6 need it running; the last cell of this notebook stops it when you are done.

## T2.3 Sign in the way Apps require

Apps servers accept OAuth only, and refuse personal access tokens [2]. This notebook's own sign-in is checked in T2.5. (`DatabricksMCPClient` also refuses it up front for app URLs, for the same reason.) For a caller with no browser, Databricks recommends a **service principal with OAuth M2M** [2].

**One-time setup** (needs an admin):
1. Create a service principal (**Settings > Identity and access**) and generate an **OAuth secret** for it. The secret value is shown **once**, when it is created.
2. Run the next cell. It asks for the service principal's **client ID** and the **OAuth secret**, **tests them against the token endpoint first**, and only then stores them in the secret scope `mcp-demo`. A wrong value stops the cell with nothing stored. On a re-run it skips the prompts.

(The CLI works too: `databricks secrets put-secret mcp-demo sp-client-id`, then `sp-client-secret`. Each command prompts for the value.)

In [0]:
from getpass import getpass

RESET_CREDENTIALS = True  # True: ask for the Application ID and secret again and replace what is stored


def secret_exists(key):
    try:
        dbutils.secrets.get(SECRET_SCOPE, key)
        return True
    except Exception:
        return False


def oauth_client(client_id, client_secret):
    """A client that signs in as the service principal with OAuth M2M."""
    return WorkspaceClient(host=w.config.host, client_id=client_id, client_secret=client_secret, auth_type="oauth-m2m")


try:
    w.secrets.create_scope(scope=SECRET_SCOPE)
except:
    print("Resource already exists")
    pass

if secret_exists("sp-client-id") and secret_exists("sp-client-secret") and not RESET_CREDENTIALS:
    print(f"Credentials already stored in scope '{SECRET_SCOPE}'. Set RESET_CREDENTIALS = True to enter them again.")
else:
    client_id = getpass("Client ID: ").strip()
    client_secret = getpass("OAuth secret: ").strip()
    try:
        oauth_client(client_id, client_secret).config.oauth_token()
    except ValueError as e:
        raise RuntimeError(f"The token endpoint rejected these credentials ({e}). Nothing was stored. See the checklist below.") from None
    w.secrets.put_secret(scope=SECRET_SCOPE, key="sp-client-id", string_value=client_id)
    w.secrets.put_secret(scope=SECRET_SCOPE, key="sp-client-secret", string_value=client_secret)
    print(f"The credentials work. Stored in scope '{SECRET_SCOPE}'.")

Resource already exists


Client ID:  [REDACTED]

OAuth secret:  [REDACTED]

The credentials work. Stored in scope 'mcp-demo'.


> **If you get `invalid_client: Client authentication failed`**: the token endpoint did not accept the pair. Check, in this order:
> 1. **The secret value.** It is shown once, at creation. Copying part of it, or something else on that page, fails exactly like this. If in doubt, generate a new secret and use that.
> 2. **The ID.** It is the service principal's **Application ID** (a UUID), not its name or its numeric id.
> 3. **Expiry.** A secret past its lifetime, or one that was deleted, fails the same way.
> 4. **The kind of credential.** A personal access token is not an OAuth secret.
>
> A permission grant on the app can succeed even when the secret is wrong, because it only needs the ID to resolve. That is why this notebook tests the secret before using it.

Now give the service principal `CAN USE` on the app, show who can use it, and sign in as the service principal:

In [0]:
SP_CLIENT_ID = dbutils.secrets.get(SECRET_SCOPE, "sp-client-id")
SP_CLIENT_SECRET = dbutils.secrets.get(SECRET_SCOPE, "sp-client-secret")

w.apps.update_permissions(
    APP_NAME,
    access_control_list=[AppAccessControlRequest(service_principal_name=SP_CLIENT_ID, permission_level=AppPermissionLevel.CAN_USE)],
)

print("Who has what on the app:")
for acl in w.apps.get_permissions(APP_NAME).access_control_list or []:
    who = acl.user_name or acl.group_name or acl.service_principal_name
    print(" ", who, "->", [p.permission_level.value for p in acl.all_permissions or []])

w_oauth = oauth_client(SP_CLIENT_ID, SP_CLIENT_SECRET)
token = w_oauth.config.oauth_token()  # raises if the secret is wrong or this is not OAuth
print("\nSigned in with OAuth as service principal", SP_CLIENT_ID, "| token type:", token.token_type)


def oauth_headers():
    """A fresh Authorization header for each request."""
    return w_oauth.config.authenticate()

Who has what on the app:
  admins -> ['CAN_MANAGE']
  pranav@platformatory.com -> ['CAN_MANAGE']
  [REDACTED] -> ['CAN_USE']

Signed in with OAuth as service principal [REDACTED] | token type: Bearer


**Look at**: the service principal in the list with `CAN USE`. Only callers on that list get through; T5 revokes it and shows the call fail.

## T2.4 Ask the server what tools it has

MCP is JSON-RPC over HTTP. This is Session 1's `tools/list`, sent by hand as the service principal. In T3 an MCP client does it for us.

In [0]:
r = rpc("tools/list", auth=oauth_headers())
show(r, limit=2500)

HTTP 200 | application/json
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "tools": [
      {
        "name": "search_wiki",
        "description": "Search the Python wiki by keywords. Returns the best-matching pages as {id, page_content, score}.",
        "inputSchema": {
          "properties": {
            "query": {
              "description": "Keywords for the topic, e.g. 'add item list'",
              "title": "Query",
              "type": "string"
            }
          },
          "required": [
            "query"
          ],
          "title": "search_wikiArguments",
          "type": "object"
        },
        "outputSchema": {
          "properties": {
            "result": {
              "items": {
                "additionalProperties": true,
                "type": "object"
              },
              "title": "Result",
              "type": "array"
            }
          },
          "required": [
            "result"
          ],
          "title": "sea

**Look at**: a `name`, a `description` and an `inputSchema`, including the parameter description from the server. The model reads exactly this. There is no `initialize` first: the server is stateless (T2.1).

> **If you get a 401 or 403**: check that the service principal is in the list above with `CAN USE`, and that the credentials are the OAuth ones. A change to app permissions can take a short while to apply. Changes to OAuth app connections can take up to 30 minutes [2].
> **If you get a 404**: the path. Our server serves `/mcp` and nothing else (T2.5).

## T2.4b Run a tool

Session 1's `tools/call`, by hand. The arguments are the ones the model would send:

In [0]:
r = rpc("tools/call", {"name": "search_wiki", "arguments": {"query": "add item list"}}, id=2, auth=oauth_headers())
show(r, limit=2500)

HTTP 200 | application/json
{
  "jsonrpc": "2.0",
  "id": 2,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "{\n  \"id\": \"loops\",\n  \"page_content\": \"`for` loops iterate over a sequence: `for n in [1, 2, 3]: print(n)`. `range(5)` gives 0 to 4. `while` repeats while a condition is true. Use `break` to stop early and `continue` to skip to the next item.\",\n  \"score\": 1,\n  \"matches\": [\n    \"item\"\n  ]\n}"
      },
      {
        "type": "text",
        "text": "{\n  \"id\": \"lists\",\n  \"page_content\": \"Lists are ordered, changeable collections: `nums = [1, 2, 3]`. Add with `nums.append(4)`, remove with `nums.remove(2)`. Index from 0: `nums[0]`. Get the length with `len(nums)`.\",\n  \"score\": 1,\n  \"matches\": [\n    \"add\"\n  ]\n}"
      }
    ],
    "structuredContent": {
      "result": [
        {
          "id": "loops",
          "page_content": "`for` loops iterate over a sequence: `for n in [1, 2, 3]: print(n)`. `range(5)` gi

In [0]:
result = r.json()["result"]
print("isError:", result["isError"])
for block in result["content"]:
    print(block["type"], "|", block["text"][:200].replace("\n", " "))
print("structuredContent keys:", list(result.get("structuredContent", {})))

isError: False
text | {   "id": "loops",   "page_content": "`for` loops iterate over a sequence: `for n in [1, 2, 3]: print(n)`. `range(5)` gives 0 to 4. `while` repeats while a condition is true. Use `break` to stop early
text | {   "id": "lists",   "page_content": "Lists are ordered, changeable collections: `nums = [1, 2, 3]`. Add with `nums.append(4)`, remove with `nums.remove(2)`. Index from 0: `nums[0]`. Get the length wi
structuredContent keys: ['result']


**Look at**: the result is a `content` list of blocks (here `text`), the same data again as `structuredContent`, and an `isError` flag. The model is given the `content`. This shape is the same as in Session 1, and T3 shows it next to what `DatabricksMCPClient` returns.

## T2.5 Break it

Session 1's server answered anyone who could reach the port. Three ways this one says no. All three ask for `tools/list`, and none gets tools back:

In [0]:
print("This notebook signs in with:", w.config.auth_type)
try:
    w.config.oauth_token()
    print("  ...and it can produce an OAuth token.")
except ValueError as e:
    print("  ...it cannot produce an OAuth token:", str(e)[:140])
print()

cases = {
    "no credentials": dict(auth=None),
    "wrong path (valid OAuth)": dict(url=MCP_URL.rsplit("/", 1)[0] + "/mcp-typo", auth=oauth_headers()),
    "notebook's own sign-in": dict(auth=w.config.authenticate()),
}
for name, kw in cases.items():
    r = rpc("tools/list", **kw)
    detail = r.headers.get("Location") or r.text
    print(f"{name:26} HTTP {r.status_code} | {detail[:80]!r}")

This notebook signs in with: runtime
  ...it cannot produce an OAuth token: OAuth tokens are not available for runtime authentication. Use an OAuth-based authentication method to access OAuth tokens.

no credentials             HTTP 401 | '{}'
wrong path (valid OAuth)   HTTP 404 | 'Not Found'
notebook's own sign-in     HTTP 401 | '{}'


**Look at**: what each one returned. What we expect, to compare with your output:

| Call | Why it fails | What to expect |
|---|---|---|
| No credentials | Nobody signed in | A redirect to a login page, or a 401. We do not follow redirects, so the `Location` shows |
| Valid OAuth, wrong path | The server serves `/mcp` only | A 404 from the server. The request got in, and nothing is served there (a local copy of this server answers `404 Not Found`) |
| The notebook's own sign-in | Apps refuse anything but OAuth, and the first lines above show what this notebook signs in with [2] | Refused, like the first row |

Exact status codes depend on the Apps front end, so go by your output, not this table.

Not shown here: a **valid OAuth token for a caller without `CAN USE`**. That one is T5, where we grant and revoke it and watch the call change.

## Left for T3

The agent still uses its **local** `search_wiki`. T3 connects it to this server with `DatabricksMCPClient`, so the model's tool list comes from `tools/list`, and then points the same client at a Databricks-provided MCP.

T3 also has its own "break it": a **bad argument** does not raise an HTTP error. The server answers `200` with `isError: true` and the message in `content`, so a client that only checks the status code never notices.

## Stop the app

Apps bill while they run [1]. Skip this cell if you are going straight on to T3.

In [0]:
STOP_APP = False  # True: stop the app now. `w.apps.start(APP_NAME)` starts it again.

if STOP_APP:
    w.apps.stop(APP_NAME)
    print("Stopping", APP_NAME)
else:
    print("Leaving", APP_NAME, "running.")

Leaving mcp-wiki-server running.


# References

Docs checked 2026-10-24. References 1 and 2 were checked again on 2026-10-25 (both pages show "Last updated Oct 7, 2026").

1. [Custom MCP servers](https://docs.databricks.com/aws/en/agents/mcp-tools/custom-mcp): Streamable HTTP, `0.0.0.0`, `DATABRICKS_APP_PORT`, the `mcp-` prefix, `/mcp`, `CAN USE`
2. [Authentication and networking for MCP](https://docs.databricks.com/aws/en/agents/mcp-tools/authentication-networking): service principal with OAuth M2M for unattended agents, no personal access tokens for Apps, `CAN USE`
3. [Deploy a Databricks app](https://docs.databricks.com/aws/dev-tools/databricks-apps/deploy): deploying from a workspace folder
4. [MCP Python SDK 2.x migration](https://py.sdk.modelcontextprotocol.io/v2/migration/#fastmcp-renamed-to-mcpserver): `FastMCP` renamed to `MCPServer`; pin `mcp<2` to keep running v1 code